In [6]:
from openpyxl.styles import PatternFill,Font,Alignment,Border,Side
import numpy as np,pandas as pd,random as rd,xlwings as xl
from collections import defaultdict
from tqdm import tqdm as tq
tq.pandas()

In [70]:
def bonn_div(x,rd,ad):
    try:
        return round(100*(-1+x[rd]/x[ad]),1)
    except:
        return 0
def sepm(x):
    x=abs(int(x))
    vt=str(x)
    vf=''
    while vt!='' and len(vt)>3:
        vf=' '+vt[-3:]+vf
        vt=vt[:-3]
    else:
        vf=vt[-3:]+vf
    return vf
def mont(x):
    x=abs(int(x))
    if x>=1e9:
        return sepm(x/1e9)+' Milliards'
    elif x>=1e6:
        return sepm(x/1e6)+' Millions'
    elif x>=1e3:
        return sepm(x/1e3)+' Mille'
    else:
        return sepm(x)
def rem_car(text,char):
    text=str(text).split(char)
    res=text[0]
    for i in text[1:]:
        res+=i
    return res
def to_hex(num,padding=2):
    hex_value = hex(num)[2:]
    if padding>0:
        hex_value=hex_value.zfill(padding)
    return hex_value.upper()
def col():
    nf=''
    for i in range(4):
        nf+=to_hex(rd.randint(0,225))
    return nf
co=[col() for i in range(5)]
def appliquer_format_excel(writer, sheet_name):
    workbook=writer.book
    worksheet=workbook[sheet_name]
    worksheet.auto_filter.ref=worksheet.dimensions# Appliquer les filtres
    header_fill=PatternFill(start_color=co[0],end_color=co[1],fill_type="solid")# Appliquer les styles pour l'en-tête
    header_font=Font(bold=True,color=co[2])
    for cell in worksheet[1]:
        cell.fill=header_fill
        cell.font=header_font
        cell.alignment=Alignment(horizontal="center",vertical="center")
    grey_fill=PatternFill(start_color=co[3],end_color=co[4],fill_type="solid")# Appliquer les styles pour le corps du tableau
    center_alignment=Alignment(horizontal="center",vertical="center")
    for row in worksheet.iter_rows(min_row=2, max_row=worksheet.max_row):
        for cell in row:
            if cell.row % 2==0:
                cell.fill=grey_fill
            cell.alignment=center_alignment
    for col in worksheet.columns:# Ajuster la largeur des colonnes
        max_length=0
        column=col[0].column_letter
        for cell in col:
            try:
                if len(str(cell.value))>max_length:
                    max_length=len(cell.value)
            except:
                pass
        adjusted_width=(max_length+2)
        worksheet.column_dimensions[column].width=adjusted_width # Appliquer une bordure autour des cellules
    thin_border=Border(left=Side(style='thin'),right=Side(style='thin'),top=Side(style='thin'),bottom=Side(style='thin'))
    for row in worksheet.iter_rows():
        for cell in row:
            cell.border=thin_border

# Nombre de lignes pas nécessaires à codifier

In [509]:
dr=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Rattachement des villes.xlsx').rename(
    columns={'SP/VILLE':'Ville'})
data=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Base_Sondage.xlsx',dtype={'NCC':str})
data.loc[:,'Effectif']=data.Effectif.apply(lambda x: 1 if x in ['Non-disponible, à renseigner',0] else x)
ddv={dr.Ville.iloc[i]:dr.Ville.iloc[i] for i in range(len(dr))}
print(data.shape,dr.shape,dr['Ville'].nunique()==len(dr))
def ville(x):
    for i in sorted(list(ddv.keys())):
        if i.upper() in str(x).upper():
            return ddv[i].upper()
            break
    else:
        return 'AUTRE'
data.loc[:,'Ville']=data.ADR_GEO_ENT.progress_apply(lambda x: ville(x))
if len(data.loc[data.Ville=='AUTRE'])>0:
    print(data.loc[data.Ville=='AUTRE'].ADR_GEO_ENT.nunique(),'villes non-codifiées')
    print(data.loc[data.Ville=='AUTRE'].ADR_GEO_ENT.unique(),len(data.loc[data.Ville=='AUTRE'])/len(data),
          len(data.loc[(data.Ville!='AUTRE')&(~data.Code_activité.isna())])/len(data))
    data=data.loc[(data.Ville!='AUTRE')&(~data.Code_activité.isna())]
else:
    print('Tout est propre, la suite peut être exécutée')
del(dr)
del(data)
del(ddv)

(41787, 9) (505, 8) True


100%|██████████████████████████████████████████████████████████████████████████| 41787/41787 [00:11<00:00, 3611.09it/s]


9021 villes non-codifiées
['COCODY RIVIERA BONOUMIN' 'PORT BOUET GONZAG  TERRE ROUGE'
 'KOUMASSI KANKANKOURA' ...
 "GRAND LAHOU /QUARTIER DALLAS -LOT N° 94 BIS-ILOT 09 BIS COTE-D'IVOIRE"
 'TREICHVILLE  A COTE DE GARE TRAIN' 'feu de sappeur pompier'] 0.36774594969727425 0.6322540503027257


# Codification des villes pour merging avec les régions et les DR

In [5]:
dr=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Rattachement des villes.xlsx').rename(
    columns={'SP/VILLE':'Ville'})
cv=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Codifications_Zones.xlsx')
cm=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Codifications_Zones.xlsx',sheet_name=1)
data=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Base_Sondage.xlsx',dtype={'NCC':str})
data.loc[:,'Effectif']=data.Effectif.apply(lambda x: 1 if x in ['Non-disponible, à renseigner',0] else x)
ddv={dr.Ville.iloc[i]:dr.Ville.iloc[i] for i in range(len(dr))}
ddc={cm.zone.iloc[i]:cm.bonne_loc.iloc[i] for i in range(len(cm))}
ddv.update({cv.zone.iloc[i]:cv.bonne_loc.iloc[i] for i in range(len(cv))})
print(data.shape,dr.shape,dr['Ville'].nunique()==len(dr))
def ville(x):
    for i in ddv.keys():
        if i.upper() in str(x).upper():
            return ddv[i].upper()
            break
    else:
        return 'AUTRE'
def commune(x):
    for i in sorted(list(ddc.keys())):
        if i.upper() in str(x).upper():
            return ddc[i].upper()
            break
    else:
        return 'ABIDJAN'
data.loc[:,'Ville']=data.ADR_GEO_ENT.progress_apply(lambda x: ville(x))
if len(data.loc[data.Ville=='AUTRE'])>0:
    bb=data.loc[data.Ville=='AUTRE',('NCC','RAISON_SOCIALE','ADR_GEO_ENT','Chiffre_Affaire','Effectif')].sort_values(
        by=['Chiffre_Affaire'],ascending=False)
    bb_=data.loc[data.Ville=='AUTRE',('NCC','RAISON_SOCIALE','ADR_GEO_ENT','Chiffre_Affaire','Effectif')].sort_values(by=['Effectif'],ascending=False)
    print(bb.loc[bb.Chiffre_Affaire>1e9,('NCC','RAISON_SOCIALE','Chiffre_Affaire')],'\n')
    print(bb.Chiffre_Affaire.sum(),round(100*bb.Chiffre_Affaire.sum()/data.Chiffre_Affaire.sum(),2))
    print(bb.Effectif.sum(),round(100*bb.Effectif.sum()/data.Effectif.sum(),2),'\n')
    print(bb_.loc[bb_.Effectif>10,('NCC','RAISON_SOCIALE','Effectif')],'\n')
    print(len(bb),"entreprises dont les localisations n'ont pu être codifiées")
    print(bb.ADR_GEO_ENT.unique(),len(bb)/len(data),len(data.loc[(data.Ville!='AUTRE')&(~data.Code_activité.isna())])/len(data))
    data.loc[(data.Ville!='AUTRE')&(~data.Code_activité.isna())].to_excel(
        'C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Base_Sondage_Codifiee.xlsx',index=False)
    data=data.loc[(data.Ville!='AUTRE')&(~data.Code_activité.isna())]
else:
    print('Tout est propre, la suite peut être exécutée')

(41780, 10) (505, 8) True


100%|██████████████████████████████████████████████████████████████████████████| 41780/41780 [00:05<00:00, 7669.33it/s]


Empty DataFrame
Columns: [NCC, RAISON_SOCIALE, Chiffre_Affaire]
Index: [] 

40607142974.257 0.11
2006 0.28 

Empty DataFrame
Columns: [NCC, RAISON_SOCIALE, Effectif]
Index: [] 

867 entreprises dont les localisations n'ont pu être codifiées
['0' 'ND' "ACHATS,TRANSFORMATIONS ET EXPORTATIONS DE L'ANACARDE"
 '0151114720' 'QUARTIER SOBA' 'BEGNERI'
 'COMMERCE MATERIELS DE LABORATION ET REACTIFS'
 'ANCIENNE GENDARMERIE CARREFOUR ADJA' 'APPROMPRON-AFEWA' 'MARCHE'
 '00/01/1900' '30/12/1899' 'SOKOURANI' '556583662'
 'QUARTIER COMMERCE-PPRINCIPALE' 'GNAPOADJI'
 'QUARTIER COMMERCE FACE BANQUE BOA' 'Quartier commerce'
 'TOUTES OPERATIONS DE TRANSIT' 'VILLAGE LOURIA' 'PARHADI'
 'Quartier commerce  lot 120C' 'QUARTIER COMMERCE FACE BICICI'
 "COTE D'IVOIRE-GNITY ECOLE" 'ESCADRON CEINTURE ALI COULIBALY' '00'
 'MARCHE EN FACE DE CALIVOIRE' '30èARRONDISSEMENT'
 'DIOULAKRO GARE SANS FRONTIERES'
 'QUARTIER COLAS 10 METRES DE LA STATION TOTAL' 'QUARTIER COMMERCE'
 'COMMERCIALISATION  SERVICES LIEES A LA CO

# Vérification de l'exhaustivité de la base de sondage codifiée

In [7]:
pt=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Echantillonnage_DR_Hors_DR.xlsx')
bds=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Base_Sondage_Codifiee.xlsx')
print(len(bds))
if len(pt.loc[~pt.NCC.isin(bds.NCC)])>0:
    print('Il manque',len(pt.loc[~pt.NCC.isin(bds.NCC)]),'NCC à la base de sondage. Ce sont :',[i for i in pt.loc[~pt.NCC.isin(bds.NCC)].NCC.unique()])
else:
    print('Aucun manquant dans la base de sondage')
    del(pt)
    del(bds)

40913
Aucun manquant dans la base de sondage


# Suite du code

In [9]:
if len(data)==len(data.merge(dr[['Ville','DR','REGION','CHEF LIEU DE REGION','STATUT REGION','STATUT VILLE_CL_Région','STATUT VILLE_CL_DR']],on='Ville',
                             how='left')):
    print('Merging Possible')
    data=data.merge(dr[['Ville','DR','REGION','CHEF LIEU DE REGION','STATUT REGION','STATUT VILLE_CL_Région','STATUT VILLE_CL_DR']],on='Ville',
                    how='left')
data.loc[:,'Chiffre_Affaire']=data.Chiffre_Affaire.apply(lambda x: float(x) if not x in ['Non-disponible',0] else 1e6)
data.loc[:,'Effectif']=data.Effectif.apply(lambda x: float(x) if not str(x) in ['nan','0'] else 1)
data.loc[:,'Taille']=data.apply(lambda x: 'GE' if x.Effectif>199 or x.Chiffre_Affaire>1e9 else 'PME',axis=1)
def act(x):
    x=str(x)[0]
    if x=='A':
        return 'Agriculture'
    elif x=='C':
        return 'Industrie Manufacturière'
    elif x=='G':
        return 'Commerce'
    elif x in 'BDEF':
        return 'Autres Industries'
    else:
        return 'Service'
def stra(x):
    if x.Activite=='Agriculture' and x.Taille=='GE':
        return 'Strate 1'
    elif x.Activite=='Agriculture' and x.Taille=='PME':
        return 'Strate 2'
    elif x.Activite=='Industrie Manufacturière' and x.Taille=='GE':
        return 'Strate 3'
    elif x.Activite=='Industrie Manufacturière' and x.Taille=='PME':
        return 'Strate 4'
    elif x.Activite=='Autres Industries' and x.Taille=='GE':
        return 'Strate 5'
    elif x.Activite=='Autres Industries' and x.Taille=='PME':
        return 'Strate 6'
    elif x.Activite=='Commerce' and x.Taille=='GE':
        return 'Strate 7'
    elif x.Activite=='Commerce' and x.Taille=='PME':
        return 'Strate 8'
    elif x.Activite=='Service' and x.Taille=='GE':
        return 'Strate 9'
    else:
        return 'Strate 10'
data.loc[:,'Activite']=data.Code_activité.apply(lambda x: act(x))
data.loc[:,'Strate']=data.apply(lambda x: stra(x),axis=1)
data.sort_values(by=['DR','Ville','Chiffre_Affaire','Effectif'],ascending=[True,True,False,False],inplace=True)
tcd=pd.pivot_table(data,index=['Ville','DR','REGION','CHEF LIEU DE REGION','STATUT REGION','STATUT VILLE_CL_Région','STATUT VILLE_CL_DR'],
               values=['Effectif','Chiffre_Affaire'],aggfunc='sum').reset_index().sort_values(by=['REGION','DR'])
tcd.loc[:,'Nombre_Entreprises']=tcd.Ville.apply(lambda x: len(data.loc[data.Ville==x]))
tcd.sort_values(by=['Nombre_Entreprises','Effectif'],ascending=False,inplace=True)
ntcd=tcd[tcd['STATUT VILLE_CL_DR']=='Ville est CL de DR'][['Ville']]
ntcd.loc[:,'Entreprises_DR']=ntcd.Ville.apply(lambda x: len(data.loc[(data.DR==x)&(data['STATUT VILLE_CL_DR']=='Ville est CL de DR')]))
ntcd.loc[:,'Entreprises_Hors_DR']=ntcd.Ville.apply(lambda x: len(data.loc[(data.DR==x)&(data['STATUT VILLE_CL_DR']!='Ville est CL de DR')]))
ntcd.loc[:,'Effectif_DR']=ntcd.Ville.apply(lambda x: data.loc[(data.DR==x)&(data['STATUT VILLE_CL_DR']=='Ville est CL de DR')].Effectif.sum())
ntcd.loc[:,'Effectif_Hors_DR']=ntcd.Ville.apply(lambda x: data.loc[(data.DR==x)&(data['STATUT VILLE_CL_DR']!='Ville est CL de DR')].Effectif.sum())
with pd.ExcelWriter('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Repartitions_DR.xlsx') as wr:
    tcd.loc[tcd['STATUT VILLE_CL_Région']=='Ville est CL de région',('Ville','DR','CHEF LIEU DE REGION','Nombre_Entreprises','Effectif')].to_excel(
        wr,sheet_name='Chefs Lieux Regions',index=False)
    appliquer_format_excel(wr,'Chefs Lieux Regions')
    tcd.loc[tcd['STATUT VILLE_CL_Région']!='Ville est CL de région',('Ville','DR','CHEF LIEU DE REGION','Nombre_Entreprises','Effectif')].to_excel(
        wr,sheet_name='Hors Chefs Lieux Regions',index=False)
    appliquer_format_excel(wr,'Hors Chefs Lieux Regions')
    tcd.loc[tcd['STATUT VILLE_CL_DR']=='Ville est CL de DR',('Ville','DR','CHEF LIEU DE REGION','Nombre_Entreprises','Effectif')].to_excel(
        wr,sheet_name='Chefs Lieux DR',index=False)
    appliquer_format_excel(wr,'Chefs Lieux DR')
    tcd.loc[tcd['STATUT VILLE_CL_DR']!='Ville est CL de DR',('Ville','DR','CHEF LIEU DE REGION','Nombre_Entreprises','Effectif')].to_excel(
        wr,sheet_name='Hors Chefs Lieux DR',index=False)
    appliquer_format_excel(wr,'Hors Chefs Lieux DR')
    pd.pivot_table(tcd,index=['DR'],values=['Nombre_Entreprises','Effectif'],aggfunc='sum').reset_index()[
    ['DR','Nombre_Entreprises','Effectif']].sort_values(by=['Nombre_Entreprises','Effectif'],ascending=False).to_excel(
        wr,sheet_name='Bilan Global',index=False)
    appliquer_format_excel(wr,'Bilan Global')
    ntcd.to_excel(wr,sheet_name='Bilan Detaille',index=False)
    appliquer_format_excel(wr,'Bilan Detaille')
print("Nombre d'entreprises avec effectif vide :",len(data.loc[data.Effectif.isna()]))

Merging Possible
Nombre d'entreprises avec effectif vide : 0


# Suite du bara

In [11]:
def bp(df,rt):
    n=1
    while df.head(n).Effectif.sum()<rt*df.Effectif.sum():
        n-=-1
    if n<10:
        return df.head(n)
    else:
        return "Pas de Poids Lourd, "+str(n)+' entreprises'
fi=pd.DataFrame()
ddt={'Strate 1':7,'Strate 3':11,'Strate 5':7,'Strate 7':11,'Strate 9':7}
for i in ddt.keys():
    fi=pd.concat([fi,data.loc[data.Strate==i,('NCC','RAISON_SOCIALE','Effectif','Strate')].sort_values(by='Effectif',ascending=False).head(ddt[i])])
fi.sort_values(by=['Strate','Effectif'],ascending=False,inplace=True)
fi=data.loc[data.NCC.isin(fi.NCC)]
ordata,data=data.copy(),data.loc[~data.NCC.isin(fi.NCC)]
fi

,NCC,Chiffre_Affaire,Effectif,RAISON_SOCIALE,ADR_GEO_ENT,Code_activité,Division,SIGLE_ENT,Forme juridique,TELEPHONE,Ville,DR,REGION,CHEF LIEU DE REGION,STATUT REGION,STATUT VILLE_CL_Région,STATUT VILLE_CL_DR,Taille,Activite,Strate
15565,9818944H,4.871646e+11,6624,STE DE DISTRIBUTION DE TOUTE M,RUE DU HAVRE TREICHVILLE ABIDJAN,G46060000,G46,STE DE DISTRIBUTION DE TOUTE M,Société Anonyme (SA),2721219000,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Commerce,Strate 7
14528,9702458W,3.483726e+11,7476,PALMCI,ZONE PORTUAIRE - BOULEVARD DE VRIDI - ABIDJAN ...,C10040003,C10,PALMCI,Société Anonyme (SA) à participation publique,2721210900,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Industrie Manufacturière,Strate 3
13482,5011806N,3.357449e+11,5123,PROSUMA-STE IVOIRIENNE,01 BP 3747 Rue Lecoeur - Plateau Abidjan,G46060000,G46,PROSUMA-STE IVOIRIENNE,Société Anonyme (SA),2721253416,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Commerce,Strate 7
3439,1205106C,3.070190e+11,4792,PFO CONSTRUCTION,"COCODY BD LATRILLE, BP 387 ABIDJAN",F41020000,F41,PFO CONSTRUCTION,Société Anonyme (SA),2722484545,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5
15475,9004996S,2.388544e+11,4579,CIE- COMPAGNIE IVOIRIENNE D'ELECTRICITE,1 AVENUE CHRISTIANI TREICHVILLE,D35010000,D35,CIE- COMPAGNIE IVOIRIENNE D'ELECTRICITE,Société Anonyme (SA) à participation publique,2721233519,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5
638,0105337H,2.244212e+11,6265,SAPH STE AFRICAINE DE PLANTATION D'HEVEAS,Treichville zone portuaire rue des galions,A01020300,A01,SAPH STE AFRICAINE DE PLANTATION D'HEVEAS,Entreprise individuelle,2721757676,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Agriculture,Strate 1
2720,1205787E,1.902259e+11,4307,PORTEO BTP,RUE DU COMMERCE ABIDJAN PLATEAU,F42000101,F42,PORTEO BTP,Société Anonyme (SA),2720333001,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5
566,0100984A,1.606710e+11,3204,SOCIETE DE DISTRIBUTION D'EAU DE LA COTE D'IVO...,Avenue Christiani 1 Treichville Abidjan Côte d...,E36000000,E36,SOCIETE DE DISTRIBUTION D'EAU DE LA COTE D'IVO...,Société Anonyme (SA) à participation publique,2721233000,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5
10024,1973269P,1.509645e+11,982,SOCOCE CI,ZONE 3 RUE DES BRASSEURS 0 0 ABIDJAN 05 Côte d...,G47010000,G47,SOCOCE CI,Société Anonyme (SA),NaN,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Commerce,Strate 7
14695,9000473M,1.485698e+11,1724,CDCI-CIE.DE DISTRIBUTION,ANGLE RUE DU PORT ABIDJAN CÔTE D'IVOIRE,G46020300,G46,CDCI-CIE.DE DISTRIBUTION,Société Anonyme (SA),2721240151,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Commerce,Strate 7


In [37]:
pre_ti_to=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Echantillonnage_DR_Hors_DR.xlsx')
pre_ti=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Echantillonnage_DR_Hors_DR.xlsx',sheet_name='Aléa')
print('Taille initiale aléa :',len(pre_ti))
pre_ti=pre_ti.loc[pre_ti.NCC.isin(pre_ti_to.NCC)]
print('Taille aléa après ajustement au total:',len(pre_ti))
pre_ti=pre_ti[[i for i in pre_ti.columns if not i in ['Ville','DR','REGION','CHEF LIEU DE REGION','STATUT REGION','STATUT VILLE_CL_Région',
                                                      'STATUT VILLE_CL_DR']]]
pre_ti.loc[:,'Ville']=pre_ti.ADR_GEO_ENT.progress_apply(lambda x: ville(x))
if len(pre_ti)==len(pre_ti.merge(dr[['Ville','DR','REGION','CHEF LIEU DE REGION','STATUT REGION','STATUT VILLE_CL_Région','STATUT VILLE_CL_DR']],
                              on='Ville',how='left')):
    print('Merging Possible')
    pre_ti=pre_ti.merge(dr[['Ville','DR','REGION','CHEF LIEU DE REGION','STATUT REGION','STATUT VILLE_CL_Région','STATUT VILLE_CL_DR']],on='Ville',
                    how='left')
    te,inte=len(pre_ti.loc[pre_ti.Ville!='ABIDJAN']),pre_ti.loc[pre_ti.Ville!='ABIDJAN']
    print("À tirer à l'intérieur :",te)
    p_=pd.pivot_table(inte,index='STATUT VILLE_CL_DR',values='Effectif',aggfunc='sum').reset_index().sort_values(by='Effectif',ascending=False)
    p_.loc[:,'Tirage']=p_.Effectif.apply(lambda x: int(round(te*x/inte.Effectif.sum())))
    print(p_)
    te=1620
    print('Reste à tirer :',te)
    p_=pd.pivot_table(data,index='STATUT VILLE_CL_DR',values='Effectif',aggfunc='sum').reset_index().sort_values(by='Effectif',ascending=False)
    p_.loc[:,'Tirage']=p_.Effectif.apply(lambda x: int(round(te*x/data.Effectif.sum())))
    print(p_)
else:
    print("Merging n'a pas eu lieu")

Taille initiale aléa : 1571
Taille aléa après ajustement au total: 1557


100%|███████████████████████████████████████████████████████████████████████████| 1557/1557 [00:00<00:00, 23516.16it/s]

Merging Possible
À tirer à l'intérieur : 314
         STATUT VILLE_CL_DR  Effectif  Tirage
1  Ville n'est pas CL de DR     17180     177
0        Ville est CL de DR     12706     131
Reste à tirer : 1620
         STATUT VILLE_CL_DR  Effectif  Tirage
0        Ville est CL de DR    469193    1431
1  Ville n'est pas CL de DR     61910     189


In [83]:
chemin='C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/'
def repart_DR(edld):
    q,r=[len(edld)//6 for i in range(6)],len(edld)%6
    if r!=0:
        for i in range(r):
            q[r]=q[r]+1
    q=list(pd.DataFrame(q).cumsum()[0])
    return {'Echantillon_1':list(edld.iloc[0:q[0]].NCC.unique()),'Echantillon_2':list(edld.iloc[q[0]:q[1]].NCC.unique()),
            'Echantillon_3':list(edld.iloc[q[1]:q[2]].NCC.unique()),'Echantillon_4':list(edld.iloc[q[2]:q[3]].NCC.unique()),
            'Echantillon_5':list(edld.iloc[q[3]:q[4]].NCC.unique()),'Echantillon_6':list(edld.iloc[q[4]:q[5]].NCC.unique())}
def dispatch(ear):
    abd,hdabd=ear.loc[(ear.Ville=='ABIDJAN')|(ear.DR=='ABIDJAN')],ear.loc[(ear.Ville!='ABIDJAN')&(ear.DR!='ABIDJAN')]
    ddrf={}
    for j in range(10):
        if len(abd.loc[abd.Strate=='Strate '+str(j+1)])>0:
            dp=repart_DR(abd.loc[abd.Strate=='Strate '+str(j+1)])
            if ddrf!={}:
                ddrf.update({'Echantillon_1':ddrf['Echantillon_1']+dp['Echantillon_1'],'Echantillon_2':ddrf['Echantillon_2']+dp['Echantillon_2'],
                             'Echantillon_3':ddrf['Echantillon_3']+dp['Echantillon_3'],'Echantillon_4':ddrf['Echantillon_4']+dp['Echantillon_4'],
                             'Echantillon_5':ddrf['Echantillon_5']+dp['Echantillon_5'],'Echantillon_6':ddrf['Echantillon_6']+dp['Echantillon_6']})
            else:
                ddrf.update({'Echantillon_1':dp['Echantillon_1'],'Echantillon_2':dp['Echantillon_2'],'Echantillon_3':dp['Echantillon_3'],
                             'Echantillon_4':dp['Echantillon_4'],'Echantillon_5':dp['Echantillon_5'],'Echantillon_6':dp['Echantillon_6']})
    for i in hdabd.DR.unique():
        for j in range(10):
            if len(hdabd.loc[(hdabd.DR==i)&(hdabd.Strate=='Strate '+str(j+1))])>0:
                dp=repart_DR(hdabd.loc[(hdabd.DR==i)&(hdabd.Strate=='Strate '+str(j+1))])
                if ddrf!={}:
                    ddrf.update({'Echantillon_1':ddrf['Echantillon_1']+dp['Echantillon_1'],'Echantillon_2':ddrf['Echantillon_2']+dp['Echantillon_2'],
                                 'Echantillon_3':ddrf['Echantillon_3']+dp['Echantillon_3'],'Echantillon_4':ddrf['Echantillon_4']+dp['Echantillon_4'],
                                 'Echantillon_5':ddrf['Echantillon_5']+dp['Echantillon_5'],'Echantillon_6':ddrf['Echantillon_6']+dp['Echantillon_6']})
                else:
                    ddrf.update({'Echantillon_1':dp['Echantillon_1'],'Echantillon_2':dp['Echantillon_2'],'Echantillon_3':dp['Echantillon_3'],
                                 'Echantillon_4':dp['Echantillon_4'],'Echantillon_5':dp['Echantillon_5'],'Echantillon_6':dp['Echantillon_6']})
    return ddrf
def tpi(df_,SP,out,st,li):# SP est le nombre d'unités à tirer
    df_=df_.loc[df_.Ville!='ABIDJAN']
    if st!=None:
        if li=='DR':
            df1=df_.loc[(df_.Strate==st)&(df_.Strate==st)&(df_['STATUT VILLE_CL_DR']=='Ville est CL de DR')]
        else:
            df1=df_.loc[(df_.Strate==st)&(df_['STATUT VILLE_CL_DR']!='Ville est CL de DR')]
    else:
        if li=='DR':
            df1=df_.loc[df_['STATUT VILLE_CL_DR']=='Ville est CL de DR']
        else:
            df1=df_.loc[df_['STATUT VILLE_CL_DR']!='Ville est CL de DR']
    if SP<len(df1):
        N=df1.Effectif.sum()# Population totale
        pas,df1.loc[:,'CumPop'],df1.loc[:,'Poids']=N/SP,df1.Effectif.cumsum(),df1.Effectif.apply(lambda x: x/N)
        if len(df1.loc[df1.Poids.isna()])>0:
            print(df1.loc[df1.Poids.isna()])
        premier=int(rd.uniform(0,1)*pas)+1# Génération du premier tirage aléatoire dans l'intervalle [1, pas]
        A=np.zeros(SP)# Étape 4 : Calcul des seuils à partir du premier tirage
        A[0]=premier
        for i in range(1,SP):
            A[i]=round(premier+i*pas)
            if A[-1]>=N:
                A[-1]=N
        df1.loc[:,'tire']=df1.CumPop.apply(lambda x: any(x>=threshold for threshold in A))# Identification des unités tirées en fonction des seuils
        return rd.choice([list(df1[df1.tire].head(SP)[out].unique()),list(np.random.choice(a=df1[out],size=SP,replace=False,p=df1.Poids))])
    else:
        return list(df1.NCC.unique())
def tdr():
    vc1={}
    for i in data.DR.unique():
        try:
            df2=pd.pivot_table(data.loc[(data['STATUT VILLE_CL_DR']!='Ville est CL de DR')&(data.DR==i)],index=['Ville','STATUT VILLE_CL_DR'],
                              values='Effectif',aggfunc='sum').reset_index().sort_values(by='Effectif',ascending=False)
            vc1.update({i:tpi(df2,1,'Ville',None,'HDR')[0]})
        except:
            print(i)
    return data.loc[data.Ville.isin(vc1.values())]
def bt():
    vc_=pd.concat([data.loc[data['STATUT VILLE_CL_DR']=='Ville est CL de DR'],tdr()])
    p=pd.pivot_table(vc_.loc[vc_['STATUT VILLE_CL_DR']=='Ville est CL de DR'],index='Strate',values='Effectif',aggfunc='sum').reset_index().sort_values(
        by='Effectif',ascending=False)
    #print('Tirage en DR :',p_.Tirage.iloc[0])
    p.loc[:,'Tirage']=p.Effectif.apply(lambda x: int(round(
        p_.Tirage.iloc[0]*x/vc_.loc[vc_['STATUT VILLE_CL_DR']=='Ville est CL de DR'].Effectif.sum())))
    d={'Strate '+str(i+1):p.loc[p.Strate=='Strate '+str(i+1)].Tirage.iloc[0] for i in range(10)}
    for i in d.keys():
        p,r=d[i]//6,d[i]%6
        if r!=0:
            d.update({i:6*(p+1)})
    df3=pd.DataFrame([d.keys(),d.values()],index=['Strate','Tirage_DR']).T
    df3.loc[:,'Reel_DR']=df3.Strate.apply(lambda x: len(vc_.loc[(vc_['STATUT VILLE_CL_DR']=='Ville est CL de DR')&(vc_.Strate==x)]))
    p=pd.pivot_table(vc_.loc[vc_['STATUT VILLE_CL_DR']!='Ville est CL de DR'],index='Strate',values='Effectif',aggfunc='sum').reset_index().sort_values(
        by='Effectif',ascending=False)
    if len(p)==10:
        p.loc[:,'Tirage']=p.Effectif.apply(lambda x: int(round(
            p_.Tirage.iloc[1]*x/vc_.loc[vc_['STATUT VILLE_CL_DR']!='Ville est CL de DR'].Effectif.sum())))
        d={'Strate '+str(i+1):p.loc[p.Strate=='Strate '+str(i+1)].Tirage.iloc[0] for i in range(10)}
        for i in d.keys():
            p,r=d[i]//6,d[i]%6
            if r!=0:
                d.update({i:6*(p+1)})
        df3.loc[:,'Tirage_HDR']=df3.Strate.apply(lambda x: d[x])
        df3.loc[:,'Reel_HDR']=df3.Strate.apply(lambda x: len(vc_.loc[(vc_['STATUT VILLE_CL_DR']!='Ville est CL de DR')&(vc_.Strate==x)]))
        df3=pd.concat([df3,pd.DataFrame(['Total']+list(df3.sum())[1:],index=df3.columns).T])
        print(df3)
        df3.loc[:,'Tirage_DR']=df3.apply(lambda x: (len(inte.loc[(inte.Strate==x.Strate)&(inte['STATUT VILLE_CL_DR']=='Ville est CL de DR')])
                                                    if x.Strate!='Total' else len(inte.loc[inte['STATUT VILLE_CL_DR']=='Ville est CL de DR'])),axis=1)
        df3.loc[:,'Tirage_HDR']=df3.apply(lambda x: (len(inte.loc[(inte.Strate==x.Strate)&(inte['STATUT VILLE_CL_DR']!='Ville est CL de DR')])
                                                     if x.Strate!='Total' else len(inte.loc[inte['STATUT VILLE_CL_DR']!='Ville est CL de DR'])),axis=1)
        return df3,vc_
def attrib(df1):
    if type(df1)==list:
        df1=da.loc[da.NCC.isin(df1)]
    df=df1.loc[(df1.Ville.isin(['ABIDJAN','ANYAMA','BINGERVILLE','SONGON']))|(df1.DR=='ABIDJAN')]
    df_=df1.loc[(~df1.Ville.isin(['ABIDJAN','ANYAMA','BINGERVILLE','SONGON']))&(df1.DR!='ABIDJAN')]
    df.loc[:,'Commune']=df.ADR_GEO_ENT.apply(lambda x: commune(x))
    dfdt=df.copy()
    df_.loc[:,'Agent']=df_.DR.apply(lambda x: 'Agent DR '+x)
    ddepc={i:list(df.loc[df.Commune==i].NCC.unique()) for i in df.Commune.unique()}
    ni,nr=[len(df)//15 for i in range(15)],len(df)%15
    if nr!=0:
        if rd.randint(0,1)==0:
            for i in range(nr):
                ni[i]=ni[i]+1
        else:
            for i in range(nr):
                ni[-(i+1)]=ni[-(i+1)]+1
    '''if nr!=0:
        for i in range(nr):
            ni[i]=ni[i]+1'''
    nda,agent_assignments={i:ni[list(cda.keys()).index(i)] for i in cda.keys()},{}
    #print(nda)
    odp=list(cda.keys())
    rd.shuffle(odp)
    for i in cda.keys():
        j=0
        como=cda[i]
        comc=[prio[como][j]]
        if len(dfdt.loc[dfdt.Commune.isin(comc)])>=nda[i]:
            agent_assignments.update({i:list(dfdt.loc[dfdt.Commune.isin(comc)].head(nda[i]).NCC.unique())})
            dfdt=dfdt.loc[~dfdt.NCC.isin(dfdt.loc[dfdt.Commune.isin(comc)].head(nda[i]).NCC)]
        else:
            while len(dfdt.loc[dfdt.Commune.isin(comc)])<nda[i]:
                j-=-1
                comc.append(prio[como][j])
            agent_assignments.update({i:list(dfdt.loc[dfdt.Commune.isin(comc)].head(nda[i]).NCC.unique())})
            dfdt=dfdt.loc[~dfdt.NCC.isin(dfdt.loc[dfdt.Commune.isin(comc)].head(nda[i]).NCC)]
    agent_assignments={tuple(j):i for i,j in agent_assignments.items()}
    #print(agent_assignments)
    def ass_agent(x):
        for i in agent_assignments.keys():
            for j in i:
                if x==j:
                    return agent_assignments[i]
    df.loc[:,'Agent']=df.NCC.apply(lambda x: ass_agent(x))
    if len(df.loc[df.Agent.isna()])>0:
        print(len(df.loc[df.Agent.isna()]),"entreprises n'ont pas été attribuées sur",len(df))
        print(df.loc[df.Agent.isna()].NCC.unique())
    rda=df.Agent.value_counts().reset_index()
    rda.loc[:,'NC']=rda.Agent.apply(lambda x: df.loc[df.Agent==x].Commune.nunique())
    print(rda.sort_values(by='NC'))
    return pd.concat([df_,df])
res=bt()
di,vc=res[0],res[1]
print(di)
da=pd.read_excel(chemin+'Echantillons/Echantillonnage_DR_Hors_DR.xlsx')
da=data.loc[data.NCC.isin(da.NCC)]
fi1=pd.read_excel(chemin+'Echantillons/Echantillonnage_DR_Hors_DR.xlsx',sheet_name='Echantillon fixe')
fi1=data.loc[data.NCC.isin(fi1.NCC)]
for i in di.Strate.unique()[:-1]:
    da=pd.concat([da,data.loc[data.NCC.isin(tpi(vc,di.loc[di.Strate==i].Tirage_DR.iloc[0],'NCC',i,'DR'))],
                  data.loc[data.NCC.isin(tpi(vc,di.loc[di.Strate==i].Tirage_HDR.iloc[0],'NCC',i,'HDR'))]])
print('Taille après tirage combiné',da.shape)
da=pd.concat([fi,fi1,da])[[i for i in data.columns if not i in ['Chiffre_Affaire','Effectif','Code_activité','Division',
                                                                'Forme juridique']]].drop_duplicates(subset='NCC')
print('Taille après dédoublonnage',da.shape)
agents=list(pd.read_excel(chemin+'Liste_agents.xlsx',header=3).Nom.unique())
cda=pd.read_excel(chemin+'Liste_agents.xlsx',header=3).sort_values(by='Commune')
cda={cda.Nom.iloc[i]:cda.Commune.iloc[i] for i in range(len(cda))}
cda={k:v for k,v in sorted(cda.items(),key=lambda item:item[1])}
prio=pd.read_excel(chemin+'Codifications_Zones.xlsx',sheet_name=2)
prio={i:tuple(prio[i].unique()) for i in prio.columns}
fii=da.loc[da.NCC.isin(pd.concat([fi,fi1]).drop_duplicates(subset='NCC').NCC)].drop_duplicates(subset='NCC')
ale=da.loc[~da.NCC.isin(fii.NCC)].drop_duplicates(subset='NCC')
di.loc[:,'Tirage_DR']=di.apply(lambda x: (len(ale.loc[(ale.Strate==x.Strate)&(ale['STATUT VILLE_CL_DR']=='Ville est CL de DR')])if x.Strate!='Total'
                                          else len(ale.loc[ale['STATUT VILLE_CL_DR']=='Ville est CL de DR'])),axis=1)
di.loc[:,'Tirage_HDR']=di.apply(lambda x: (len(ale.loc[(ale.Strate==x.Strate)&(ale['STATUT VILLE_CL_DR']!='Ville est CL de DR')])if x.Strate!='Total'
                                           else len(ale.loc[ale['STATUT VILLE_CL_DR']!='Ville est CL de DR'])),axis=1)
print(di)
nom_de_fichier='Echantillon_Interieur_Sature.xlsx'
with pd.ExcelWriter(chemin+'Echantillons/'+nom_de_fichier) as wr:
    da.to_excel(wr,sheet_name='Tirage global',index=False)
    appliquer_format_excel(wr,'Tirage global')
    ale.to_excel(wr,sheet_name='Aléa',index=False)
    appliquer_format_excel(wr,'Aléa')
    attrib(fii).to_excel(wr,sheet_name='Echantillon fixe',index=False)
    appliquer_format_excel(wr,'Echantillon fixe')
    repartition=dispatch(ale)
    for i in range(6):
        attrib(repartition['Echantillon_'+str(i+1)]).to_excel(wr,sheet_name='Echantillon_'+str(i+1),index=False)
        appliquer_format_excel(wr,'Echantillon_'+str(i+1))
ech=pd.read_excel(chemin+'Echantillons/'+nom_de_fichier,sheet_name=None)
if not len(ech['Tirage global'])==(len(ech['Echantillon fixe'])+len(ech['Echantillon_1'])+len(ech['Echantillon_2'])+len(ech['Echantillon_3'])+
                                   len(ech['Echantillon_4'])+len(ech['Echantillon_5'])+len(ech['Echantillon_6'])):
    print(len(ech['Tirage global']),(len(ech['Echantillon fixe'])+len(ech['Echantillon_1'])+len(ech['Echantillon_2'])+len(ech['Echantillon_3'])+
                                     len(ech['Echantillon_4'])+len(ech['Echantillon_5'])+len(ech['Echantillon_6'])))
else:
    print('Identique')
    tg=pd.concat([ech['Echantillon fixe'],ech['Echantillon_1'],ech['Echantillon_2'],ech['Echantillon_3'],ech['Echantillon_4'],ech['Echantillon_5'],
                  ech['Echantillon_6']]).sort_values(by='Agent').drop_duplicates(subset='NCC')
    if tg.NCC.value_counts().reset_index()['count'].iloc[0]>1:
        print('Présence de doublons')
    else:
        print('Plus de doublons')
        with pd.ExcelWriter(chemin+'Echantillons/'+nom_de_fichier) as wr:
            tg.to_excel(wr,sheet_name='Tirage global',index=False)
            appliquer_format_excel(wr,'Tirage global')
            ech['Aléa'].to_excel(wr,sheet_name='Aléa',index=False)
            appliquer_format_excel(wr,'Aléa')
            ech['Echantillon fixe'].sort_values(by='Agent').to_excel(wr,sheet_name='Echantillon fixe',index=False)
            appliquer_format_excel(wr,'Echantillon fixe')
            for i in range(6):
                ech['Echantillon_'+str(i+1)].sort_values(by='Agent').to_excel(wr,sheet_name='Echantillon_'+str(i+1),index=False)
                appliquer_format_excel(wr,'Echantillon_'+str(i+1))
ech=pd.read_excel(chemin+'Echantillons/'+nom_de_fichier,sheet_name=None)
te=pd.DataFrame()
for i in list(ech.keys()):
    if 'Agent' in ech[i].columns:
        fic=ech[i]
        if len(te)==0:
            te=fic.loc[fic.Ville.isin(['ABIDJAN','ANYAMA','BINGERVILLE','SONGON'])].Agent.value_counts().reset_index().rename(
                columns={'count':'Entreprises '+i})
            te.loc[:,'Communes '+i]=te.Agent.apply(lambda x: fic.loc[fic.Agent==x].Commune.nunique())
        else:
            tem=fic.loc[fic.Ville.isin(['ABIDJAN','ANYAMA','BINGERVILLE','SONGON'])].Agent.value_counts().reset_index().rename(
                columns={'count':'Entreprises '+i})
            tem.loc[:,'Communes '+i]=tem.Agent.apply(lambda x: fic.loc[fic.Agent==x].Commune.nunique())
            te=te.merge(tem,on='Agent',how='left')
te.sort_values(by='Communes Tirage global',ascending=False,inplace=True)
te.to_excel(chemin+'Repartition_Agents.xlsx',index=False)
te

      Strate Tirage_DR Reel_DR Tirage_HDR Reel_HDR
0   Strate 1        30      29         30        5
1   Strate 2         6     208          6       30
2   Strate 3       258     296         18       12
3   Strate 4        48    2164          6      124
4   Strate 5       168     348         12        5
5   Strate 6        54    3671         12      171
6   Strate 7       132     761         30       41
7   Strate 8       120   11615         24      764
8   Strate 9       396     629         30       17
9  Strate 10       246   13335         42      594
0      Total      1458   33056        210     1763
      Strate Tirage_DR Reel_DR Tirage_HDR Reel_HDR
0   Strate 1        14      29         24        5
1   Strate 2         8     208         13       30
2   Strate 3        17     296         27       12
3   Strate 4         1    2164          8      124
4   Strate 5        13     348          6        5
5   Strate 6         2    3671          6      171
6   Strate 7        10     761 

C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pas,df1.loc[:,'CumPop'],df1.loc[:,'Poids']=N/SP,df1.Effectif.cumsum(),df1.Effectif.apply(lambda x: x/N)
C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pas,df1.loc[:,'CumPop'],df1.loc[:,'Poids']=N/SP,df1.Effectif.cumsum(),df1.Effectif.apply(lambda x: x/N)
C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\26802

Taille après tirage combiné (1825, 20)
Taille après dédoublonnage (1796, 15)
      Strate Tirage_DR Reel_DR Tirage_HDR Reel_HDR
0   Strate 1        26      29          9        5
1   Strate 2        11     208         17       30
2   Strate 3       192     296         23       12
3   Strate 4        42    2164         13      124
4   Strate 5       159     348          6        5
5   Strate 6        59    3671          7      171
6   Strate 7       176     761         38       41
7   Strate 8       136   11615         58      764
8   Strate 9       387     629         24       17
9  Strate 10       271   13335         94      594
0      Total      1459   33056        289     1763


C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,'Commune']=df.ADR_GEO_ENT.apply(lambda x: commune(x))
C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:114: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_.loc[:,'Agent']=df_.DR.apply(lambda x: 'Agent DR '+x)
C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:151: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slic

                         Agent  count  NC
0      GNANCHOU Franck Richard      3   1
1   CAMARA Mahamoudou Emmanuel      3   1
2              DIAKITE Brahima      3   1
3              KRA Noak Franck      3   1
4        FADIGA Abdoul Dramane      3   1
6                SILUE Ousmane      3   1
7           DOFFOU René Junior      3   1
8      COULIBALY About Yacouba      3   1
11     TCHESSIAN Roland-Franck      3   1
12        OTCHOHO Ndory Gérard      3   1
13             DIOMANDE Moussa      2   1
14          AKE Akalé Florence      2   1
5     TAHE Nanhonkado Aristide      3   2
9      BAHAN Gonlégbé Wilfried      3   2
10           KODJO Kouamé Tano      3   3
                         Agent  count  NC
0   CAMARA Mahamoudou Emmanuel     14   1
1     TAHE Nanhonkado Aristide     14   1
2                SILUE Ousmane     14   1
3      COULIBALY About Yacouba     14   1
4           AKE Akalé Florence     14   1
6      GNANCHOU Franck Richard     14   1
12             DIAKITE Brahima    

...

C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,'Commune']=df.ADR_GEO_ENT.apply(lambda x: commune(x))
C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:114: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_.loc[:,'Agent']=df_.DR.apply(lambda x: 'Agent DR '+x)
C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:151: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slic

                         Agent  count  NC
0     TAHE Nanhonkado Aristide     15   1
5              DIAKITE Brahima     14   1
6              KRA Noak Franck     14   1
9           AKE Akalé Florence     14   1
10          DOFFOU René Junior     14   1
11           KODJO Kouamé Tano     14   1
12        OTCHOHO Ndory Gérard     14   1
1      BAHAN Gonlégbé Wilfried     15   2
7   CAMARA Mahamoudou Emmanuel     14   2
8      GNANCHOU Franck Richard     14   2
13     COULIBALY About Yacouba     14   2
3        FADIGA Abdoul Dramane     15   3
4              DIOMANDE Moussa     14   3
14               SILUE Ousmane     14   3
2      TCHESSIAN Roland-Franck     15   4


C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,'Commune']=df.ADR_GEO_ENT.apply(lambda x: commune(x))
C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:114: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_.loc[:,'Agent']=df_.DR.apply(lambda x: 'Agent DR '+x)
C:\Users\defaultuser0\AppData\Local\Temp\ipykernel_27204\2680251973.py:151: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slic

                         Agent  count  NC
5      GNANCHOU Franck Richard     14   1
6              DIOMANDE Moussa     14   1
9              DIAKITE Brahima     14   1
10     COULIBALY About Yacouba     14   1
11          DOFFOU René Junior     14   1
13        OTCHOHO Ndory Gérard     14   1
14               SILUE Ousmane     14   1
0   CAMARA Mahamoudou Emmanuel     15   2
1        FADIGA Abdoul Dramane     15   2
4      TCHESSIAN Roland-Franck     15   2
7           AKE Akalé Florence     14   2
8            KODJO Kouamé Tano     14   2
12             KRA Noak Franck     14   2
2      BAHAN Gonlégbé Wilfried     15   4
3     TAHE Nanhonkado Aristide     15   5
Identique
Plus de doublons


,Agent,Entreprises Tirage global,Communes Tirage global,Entreprises Echantillon fixe,Communes Echantillon fixe,Entreprises Echantillon_1,Communes Echantillon_1,Entreprises Echantillon_2,Communes Echantillon_2,Entreprises Echantillon_3,Communes Echantillon_3,Entreprises Echantillon_4,Communes Echantillon_4,Entreprises Echantillon_5,Communes Echantillon_5,Entreprises Echantillon_6,Communes Echantillon_6
0,FADIGA Abdoul Dramane,90,8,3,1,14,2,14,2,14,2,15,2,15,3,15,2
1,TCHESSIAN Roland-Franck,90,8,3,1,14,4,14,4,14,2,15,6,15,4,15,2
2,BAHAN Gonlégbé Wilfried,89,6,3,2,14,5,14,1,14,1,14,1,15,2,15,4
3,TAHE Nanhonkado Aristide,89,6,3,2,14,1,14,1,14,1,15,1,15,1,14,5
10,GNANCHOU Franck Richard,87,6,3,1,14,1,14,2,14,4,14,3,14,2,14,1
14,SILUE Ousmane,87,6,3,1,14,1,14,1,14,2,14,2,14,3,14,1
6,DIOMANDE Moussa,88,4,2,1,14,2,15,2,15,3,14,2,14,3,14,1
12,KRA Noak Franck,87,4,3,1,14,2,14,1,14,1,14,1,14,1,14,2
4,AKE Akalé Florence,88,3,2,1,14,1,15,3,15,3,14,1,14,1,14,2
5,CAMARA Mahamoudou Emmanuel,88,3,3,1,14,1,14,1,14,2,14,2,14,2,15,2


In [10]:
data=pd.read_excel('C:/Users/USER/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Echantillons/Echantillon_Interieur_Sature.xlsx')
ext=[i for i in data.Agent.unique() if 'DR' in i]
data.loc[~data.Agent.isin(ext)]

,NCC,RAISON_SOCIALE,ADR_GEO_ENT,SIGLE_ENT,TELEPHONE,Ville,DR,REGION,CHEF LIEU DE REGION,STATUT REGION,STATUT VILLE_CL_Région,STATUT VILLE_CL_DR,Taille,Activite,Strate,Agent,Commune
0,1715838X,SOCIETE DE DEVELOPPEMENT DU CAOUTCHOUC IVOIRIEN,"ABIDJAN-COCODY-RUE LYCEE TECHNIQUE, S/N RESIDE...",SOCIETE DE DEVELOPPEMENT DU CAOUTCHOUC IVOIRIEN,2722406135,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Agriculture,Strate 1,AKE Akalé Florence,COCODY
1,0815218H,OFFICE NATIONAL DE L'EAU POTABLE,COCODY II PLATEAUX VALLONS RUE J 93,OFFICE NATIONAL DE L'EAU POTABLE,2722514300,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5,AKE Akalé Florence,COCODY
2,1942796Z,TECHNIC BETON,KM 20 AUTOROUTE DU NORD ANYAMA ALLOKOI 08 BP 4...,TECHNIC BETON,0707111111,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Industrie Manufacturière,Strate 3,AKE Akalé Florence,ABOBO
3,4104288G,SOCIETE DE DISTRIBUTION ET DESTRAVAUX,ABIDJAN -COCODY -RIVIERA - AKOUEDO PALMERAIE,SOCIETE DE DISTRIBUTION ET DESTRAVAUX,0778542914,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5,AKE Akalé Florence,COCODY
4,9801444C,SOCIETE CONCESSIONNAIRE DU PONT RIVIERA MARCORY,"Bâtiment opérationnel, sis à la barrière de pé...",SOCIETE CONCESSIONNAIRE DU PONT RIVIERA MARCORY,2722427300,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5,AKE Akalé Florence,COCODY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1791,1715354H,TERMINAL INDUSTRIEL POLYVALENT DE SAN PEDRO,"San Pedro, Boulevard Zone d¿extension du Port,...",TERMINAL INDUSTRIEL POLYVALENT DE SAN PEDRO,2722505950 / 2722505905 / 2722505905 / 2722505906,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Service,Strate 9,TCHESSIAN Roland-Franck,ABIDJAN
1792,0332868Q,INSTITUT TRAORE ADAMA,KOUMASSI GRAND MARCHE,INSTITUT TRAORE ADAMA,0504102222,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,PME,Service,Strate 10,TCHESSIAN Roland-Franck,KOUMASSI
1793,1321738F,ATLANTIC BUSINESS INTERNATIONA,TOUR AMCI 15 AVENUE JOSEPH ANOMA COTE D'IVOIRE,ATLANTIC BUSINESS INTERNATIONA,0103141400,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Service,Strate 9,TCHESSIAN Roland-Franck,ABIDJAN
1794,0027449E,ZAHHAR EPSE FAZAA Marie Therese,"IMMEUBLE LE MASSAI, Bd VGE, MARCORY/ABIDJAN",ZAHHAR EPSE FAZAA Marie Therese,2721246056,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,PME,Service,Strate 10,TCHESSIAN Roland-Franck,MARCORY


# Comparaisons des Compositions des Echantillons et de la Population Totale

In [84]:
ech=pd.read_excel(chemin+'Echantillons/'+nom_de_fichier,sheet_name=None)
for i in list(ech.keys()):
    print(i)
    fit=ech[i]
    print('Abidjan',len(fit.loc[fit.Ville.isin(['ABIDJAN','ANYAMA','BINGERVILLE','SONGON'])]))
    print('Intérieur',len(fit.loc[~fit.Ville.isin(['ABIDJAN','ANYAMA','BINGERVILLE','SONGON'])]))
    for j in range(10):
        print('Strate ',str(j+1),len(fit.loc[fit.Strate=='Strate '+str(j+1)]))
    print('\n')

Tirage global
Abidjan 1319
Intérieur 477
Strate  1 42
Strate  2 28
Strate  3 228
Strate  4 55
Strate  5 172
Strate  6 66
Strate  7 226
Strate  8 194
Strate  9 420
Strate  10 365


Aléa
Abidjan 1276
Intérieur 472
Strate  1 35
Strate  2 28
Strate  3 215
Strate  4 55
Strate  5 165
Strate  6 66
Strate  7 214
Strate  8 194
Strate  9 411
Strate  10 365


Echantillon fixe
Abidjan 43
Intérieur 5
Strate  1 7
Strate  2 0
Strate  3 13
Strate  4 0
Strate  5 7
Strate  6 0
Strate  7 12
Strate  8 0
Strate  9 9
Strate  10 0


Echantillon_1
Abidjan 210
Intérieur 44
Strate  1 3
Strate  2 1
Strate  3 32
Strate  4 7
Strate  5 25
Strate  6 10
Strate  7 30
Strate  8 26
Strate  9 65
Strate  10 55


Echantillon_2
Abidjan 213
Intérieur 64
Strate  1 4
Strate  2 1
Strate  3 34
Strate  4 11
Strate  5 28
Strate  6 11
Strate  7 32
Strate  8 28
Strate  9 70
Strate  10 58


Echantillon_3
Abidjan 212
Intérieur 72
Strate  1 5
Strate  2 7
Strate  3 34
Strate  4 9
Strate  5 31
Strate  6 12
Strate  7 36
Strate  8 28
Strat

In [87]:
ntcd=pd.DataFrame(data.DR.unique(),columns=['DR'])
ntcd.loc[:,'Entreprises_DR']=ntcd.DR.apply(lambda x: len(ordata.loc[(ordata.DR==x)&(ordata['STATUT VILLE_CL_DR']=='Ville est CL de DR')]))
ntcd.loc[:,'Poids_Entreprises_DR']=ntcd.Entreprises_DR.apply(lambda x: str(int(round(100*x/ntcd.Entreprises_DR.sum())))+'%')
ntcd.loc[:,'Entreprises_Hors_DR']=ntcd.DR.apply(lambda x: len(ordata.loc[(ordata.DR==x)&(ordata['STATUT VILLE_CL_DR']!='Ville est CL de DR')]))
ntcd.loc[:,'Poids_Entreprises_Hors_DR']=ntcd.Entreprises_Hors_DR.apply(lambda x: str(int(round(100*x/ntcd.Entreprises_Hors_DR.sum())))+'%')
ntcd.loc[:,'Effectif_DR']=ntcd.DR.apply(lambda x: ordata.loc[(ordata.DR==x)&(ordata['STATUT VILLE_CL_DR']=='Ville est CL de DR')].Effectif.sum())
ntcd.loc[:,'Poids_Effectif_DR']=ntcd.Effectif_DR.apply(lambda x: str(int(round(100*x/ntcd.Effectif_DR.sum())))+'%')
ntcd.loc[:,'Effectif_Hors_DR']=ntcd.DR.apply(lambda x: ordata.loc[(ordata.DR==x)&(ordata['STATUT VILLE_CL_DR']!='Ville est CL de DR')].Effectif.sum())
ntcd.loc[:,'Poids_Effectif_Hors_DR']=ntcd.Effectif_Hors_DR.apply(lambda x: str(int(round(100*x/ntcd.Effectif_Hors_DR.sum())))+'%')
data_=data.loc[data.NCC.isin(da.NCC)].copy()
ntcd_=pd.DataFrame(data_.DR.unique(),columns=['DR'])
ntcd_.loc[:,'Entreprises_DR']=ntcd_.DR.apply(lambda x: len(data_.loc[(data_.DR==x)&(data_['STATUT VILLE_CL_DR']=='Ville est CL de DR')]))
ntcd_.loc[:,'Poids_Entreprises_DR']=ntcd_.Entreprises_DR.apply(lambda x: str(int(round(100*x/ntcd_.Entreprises_DR.sum())))+'%')
ntcd_.loc[:,'Entreprises_Hors_DR']=ntcd_.DR.apply(lambda x: len(data_.loc[(data_.DR==x)&(data_['STATUT VILLE_CL_DR']!='Ville est CL de DR')]))
ntcd_.loc[:,'Poids_Entreprises_Hors_DR']=ntcd_.Entreprises_Hors_DR.apply(lambda x: str(int(round(100*x/ntcd_.Entreprises_Hors_DR.sum())))+'%')
ntcd_.loc[:,'Effectif_DR']=ntcd_.DR.apply(lambda x: data_.loc[(data_.DR==x)&(data_['STATUT VILLE_CL_DR']=='Ville est CL de DR')].Effectif.sum())
ntcd_.loc[:,'Poids_Effectif_DR']=ntcd_.Effectif_DR.apply(lambda x: str(int(round(100*x/ntcd_.Effectif_DR.sum())))+'%')
ntcd_.loc[:,'Effectif_Hors_DR']=ntcd_.DR.apply(lambda x: data_.loc[(data_.DR==x)&(data_['STATUT VILLE_CL_DR']!='Ville est CL de DR')].Effectif.sum())
ntcd_.loc[:,'Poids_Effectif_Hors_DR']=ntcd_.Effectif_Hors_DR.apply(lambda x: str(int(round(100*x/ntcd_.Effectif_Hors_DR.sum())))+'%')
ntcd1=pd.DataFrame(['Strate '+str(u+1) for u in range(10)],columns=['Strate'])
ntcd1.loc[:,'Entreprises_DR']=ntcd1.Strate.apply(lambda x: len(ordata.loc[(ordata.Strate==x)&(ordata['STATUT VILLE_CL_DR']=='Ville est CL de DR')]))
ntcd1.loc[:,'Poids_Entreprises_DR']=ntcd1.Entreprises_DR.apply(lambda x: str(int(round(100*x/ntcd1.Entreprises_DR.sum())))+'%')
ntcd1.loc[:,'Entreprises_Hors_DR']=ntcd1.Strate.apply(lambda x: len(ordata.loc[(ordata.Strate==x)&(
    ordata['STATUT VILLE_CL_DR']!='Ville est CL de DR')]))
ntcd1.loc[:,'Poids_Entreprises_Hors_DR']=ntcd1.Entreprises_Hors_DR.apply(lambda x: str(int(round(100*x/ntcd1.Entreprises_Hors_DR.sum())))+'%')
ntcd1.loc[:,'Effectif_DR']=ntcd1.Strate.apply(lambda x: ordata.loc[(ordata.Strate==x)&(
    ordata['STATUT VILLE_CL_DR']=='Ville est CL de DR')].Effectif.sum())
ntcd1.loc[:,'Poids_Effectif_DR']=ntcd1.Effectif_DR.apply(lambda x: str(int(round(100*x/ntcd1.Effectif_DR.sum())))+'%')
ntcd1.loc[:,'Effectif_Hors_DR']=ntcd1.Strate.apply(
    lambda x: ordata.loc[(ordata.Strate==x)&(ordata['STATUT VILLE_CL_DR']!='Ville est CL de DR')].Effectif.sum())
ntcd1.loc[:,'Poids_Effectif_Hors_DR']=ntcd1.Effectif_Hors_DR.apply(lambda x: str(int(round(100*x/ntcd1.Effectif_Hors_DR.sum())))+'%')
ntcd1_=pd.DataFrame(['Strate '+str(u+1) for u in range(10)],columns=['Strate'])
ntcd1_.loc[:,'Entreprises_DR']=ntcd1_.Strate.apply(lambda x: len(data_.loc[(data_.Strate==x)&(data_['STATUT VILLE_CL_DR']=='Ville est CL de DR')]))
ntcd1_.loc[:,'Poids_Entreprises_DR']=ntcd1_.Entreprises_DR.apply(lambda x: str(int(round(100*x/ntcd1_.Entreprises_DR.sum())))+'%')
ntcd1_.loc[:,'Entreprises_Hors_DR']=ntcd1_.Strate.apply(lambda x: len(data_.loc[(data_.Strate==x)&(data_['STATUT VILLE_CL_DR']!='Ville est CL de DR')]))
ntcd1_.loc[:,'Poids_Entreprises_Hors_DR']=ntcd1_.Entreprises_Hors_DR.apply(lambda x: str(int(round(100*x/ntcd1_.Entreprises_Hors_DR.sum())))+'%')
ntcd1_.loc[:,'Effectif_DR']=ntcd1_.Strate.apply(
    lambda x: data_.loc[(data_.Strate==x)&(data_['STATUT VILLE_CL_DR']=='Ville est CL de DR')].Effectif.sum())
ntcd1_.loc[:,'Poids_Effectif_DR']=ntcd1_.Effectif_DR.apply(lambda x: str(int(round(100*x/ntcd1_.Effectif_DR.sum())))+'%')
ntcd1_.loc[:,'Effectif_Hors_DR']=ntcd1_.Strate.apply(
    lambda x: data_.loc[(data_.Strate==x)&(data_['STATUT VILLE_CL_DR']!='Ville est CL de DR')].Effectif.sum())
ntcd1_.loc[:,'Poids_Effectif_Hors_DR']=ntcd1_.Effectif_Hors_DR.apply(lambda x: str(int(round(100*x/ntcd1_.Effectif_Hors_DR.sum())))+'%')
with pd.ExcelWriter('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Repartitions_Echantillon.xlsx') as wr:
    ntcd.to_excel(wr,sheet_name='Base de Sondage par DR',index=False)
    appliquer_format_excel(wr,'Base de Sondage par DR')
    ntcd_.to_excel(wr,sheet_name='Echantillon par DR',index=False)
    appliquer_format_excel(wr,'Echantillon par DR')
    ntcd1.to_excel(wr,sheet_name='Base de Sondage par Strate',index=False)
    appliquer_format_excel(wr,'Base de Sondage par Strate')
    ntcd1_.to_excel(wr,sheet_name='Echantillon par Strate',index=False)
    appliquer_format_excel(wr,'Echantillon par Strate')

# Branche d'Activité et Trimestre d'Entrée

In [78]:
tde=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Echantillonnage_DR_Hors_DR.xlsx',sheet_name=None)
ar=['Echantillon fixe','Echantillon_1','Echantillon_2','Echantillon_3','Echantillon_4','Echantillon_5','Echantillon_6']
tde={i:tde[i] for i in tde.keys() if i in ar}
tde.update({i:tuple(tde[i].NCC.unique()) for i in tde.keys()})
tde={j:i for i,j in tde.items()}
dda={'A':'Agriculture, Sylviculture, Pêche','B':'Activités extractives','C':'Activités de fabrication',
     'D':'Production et distribution d\'électricité et de gaz',
     'E':'Production et distribution d\'eau, assainissement ; traitement des déchets et dépollution','F':'Construction','G':'Commerce',
     'H':'Transport et Entreposage','I':'Hébergement et Restauration','J':'Information et Communication','K':'Activités Financières et d\'Assurance',
     'L':'Activités Immobilières','M':'Activités Spécialisées, Scientifiques et Techniques','N':'Activités de Services de soutien et de Bureau',
     'O':'Activités d\'Administration Publique','P':'Enseignement','Q':'Activités pour la santé humaine et l\'action sociale',
     'R':'Activités artistiques, sportives et récréatives','S':'Autres Activités de Services n.c.a.','T':'Activités Spéciales des Ménages',
     'U':'Activités des organisations extraterritoriales'}
rpd=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Repartition des entreprises par DR.xlsx',sheet_name=None)
def tden(x):
    for i in tde.keys():
        if x in i:
            return tde[i]
with pd.ExcelWriter('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Repartitions_Entreprises_DR.xlsx') as wr:
    for i in rpd.keys():
        dat=rpd[i]
        dat.loc[:,'Branche_Activite']=dat.Division.apply(lambda x: dda[x[0]])
        dat.loc[:,'Entree']=dat.NCC.apply(lambda x: tden(x))
        dat.sort_values(by=['Branche_Activite','Entree']).to_excel(wr,sheet_name=i,index=False)
        appliquer_format_excel(wr,i)

C:\Anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:79: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
C:\Anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:79: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


# Entreprises Par Agent

In [809]:
tde=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Echantillonnage_Renforce.xlsx',sheet_name=None)
ar=['Echantillon fixe','Echantillon_1','Echantillon_2','Echantillon_3','Echantillon_4','Echantillon_5','Echantillon_6']
tde={i:tde[i] for i in tde.keys() if i in ar}
tde.update({i:tuple(tde[i].NCC.unique()) for i in tde.keys()})
tde={j:i for i,j in tde.items()}
tir=pd.read_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Echantillonnage_Renforce.xlsx')
def tden(x):
    for i in tde.keys():
        if x in i:
            return tde[i]
tir=tir.loc[tir.Ville=='ABIDJAN']
tir.loc[:,'Entree']=tir.NCC.apply(lambda x: tden(x))
tir

,NCC,Chiffre_Affaire,Effectif,RAISON_SOCIALE,ADR_GEO_ENT,Code_activité,Division,SIGLE_ENT,Forme juridique,TELEPHONE,...,REGION,CHEF LIEU DE REGION,STATUT REGION,STATUT VILLE_CL_Région,STATUT VILLE_CL_DR,Taille,Activite,Strate,Agent,Entree
320,6013175P,2.661099e+12,798,SIR ( SOCIETE IVOIRIENNE RAFFINAGE ),"VRIDI, ROUTE DE PETIT BASSAM - 01 BP 1269 AB...",C19010000,C19,SIR ( SOCIETE IVOIRIENNE RAFFINAGE ),Société Anonyme (SA) à participation publique,2721237028,...,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Industrie Manufacturière,Strate 3,TAHE Nanhonkado Aristide,Echantillon_1
321,7603142C,5.731301e+11,342,TOTALENERGIES MARKETING CÔTE D'IVOIRE,IMMEUBLE RIVE GAUCHE 100 ZONE 3 TREICHVILLE AB...,G46040100,G46,TOTALENERGIES MARKETING CÔTE D'IVOIRE,Société Anonyme (SA),2721222323,...,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Commerce,Strate 7,OTCHOHO Ndory Gérard,Echantillon_1
322,9606123E,5.659247e+11,1586,ORANGE COTE D'IVOIRE SA,0 0 0 ABIDJAN Côte d'Ivoire,J61000000,J61,ORANGE COTE D'IVOIRE SA,Société Anonyme (SA) à participation publique,0778767473,...,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Service,Strate 9,FADIGA Abdoul Dramane,Echantillon_1
323,7602349S,5.186430e+11,526,PETROCI HOLDING,"Immeuble LES HEVEAS - 14, Boulevard CARDE - Pl...",B06000200,B06,PETROCI HOLDING,Société Anonyme (SA) à participation publique,2720202528,...,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5,AKE Akalé Florence,Echantillon_1
324,0100690X,4.889018e+11,164,VIVO ENERGY CI,Rue des pétroliers Zone Industrielle de Vridi ...,G47020800,G47,VIVO ENERGY CI,Société Anonyme (SA) à participation publique,2721752727,...,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Commerce,Strate 7,COULIBALY About Yacouba,Echantillon_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2185,0332179M,1.000000e+00,5,EGLISE PROTESTANTE BAPTISTE OEUVRES ET MISSION IN,YOPOUGON KOUTE,S94030100,S94,EGLISE PROTESTANTE BAPTISTE OEUVRES ET MISSION IN,Société en Nom Collectif (SNC),0708854269,...,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,PME,Service,Strate 10,FADIGA Abdoul Dramane,None
2186,1643612C,1.000000e+00,3,IFODIS,"COCODY - 2 PLATEAUX, Derrière Mosquée AGHIEN, ...",F41010000,F41,IFODIS,Société à Responsabilité Limitée (SARL),0707017183,...,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,PME,Autres Industries,Strate 6,DOFFOU René Junior,None
2187,0040684K,1.000000e+00,1,OLAM COCOA IVOIRE SA (EX ADM COCOA),ZONE INDUSTRIELLE 0 PORT BOUET ABIDJAN Côte d'...,G46020100,G46,OLAM COCOA IVOIRE SA (EX ADM COCOA),Société Anonyme (SA),2721219677,...,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,PME,Commerce,Strate 8,AKE Akalé Florence,None
2188,0201099P,1.000000e+00,1,ROYAL OIL,MARCORY BIETRY ...,G47020800,G47,ROYAL OIL,Société Anonyme (SA),2721251445,...,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,PME,Commerce,Strate 8,COULIBALY About Yacouba,None


In [ ]:
with pd.ExcelWriter('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Agents/Repartition_Agents.xlsx') as wr:
    for i in tir.Agent.unique():
        dat=rpd[i]
        dat.loc[:,'Branche_Activite']=dat.Division.apply(lambda x: dda[x[0]])
        dat.loc[:,'Entree']=dat.NCC.apply(lambda x: tden(x))
        dat.sort_values(by=['Branche_Activite','Entree']).to_excel(wr,sheet_name=i,index=False)
        tir.loc[tir.Agent==i].to_excel('C:/Users/defaultuser0/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Agents/'+i+'.xlsx',index=False)
        tir.loc[tir.Agent==i].to_excel(wr,sheet_name=i,index=False)
        appliquer_format_excel(wr,i)
del(tir)

# Tableaux Croisés Note Méthodologique

In [44]:
data=pd.read_excel('C:/Users/USER/OneDrive/Documents/PHAS/Enquetes/Enquete Emploi/Excel/Echantillons/Echantillon_Interieur_Sature.xlsx',sheet_name=None)
da=data[list(data.keys())[0]]
lde=list(data.keys())[3:]
print(lde)
def entre(NCC):
    for i in lde:
        if NCC in data[i].NCC.unique():
            return i
    else:
        return lde[0]
da['Entrée']=da.NCC.apply(lambda x: entre(x))
source=pd.read_excel(r"C:\Users\USER\OneDrive\Documents\PHAS\Centrales\2024\Reprtoire_2024_Codif_VILLE_09092024.xlsx")
if len(da)==len(da.merge(source,on='NCC',how='left')):
    da=da.merge(source[['NCC','Effectif_Employé','CA22_corrige']],on='NCC',how='left')
da

['Echantillon_1', 'Echantillon_2', 'Echantillon_3', 'Echantillon_4', 'Echantillon_5', 'Echantillon_6']


,NCC,RAISON_SOCIALE,ADR_GEO_ENT,SIGLE_ENT,TELEPHONE,Ville,DR,REGION,CHEF LIEU DE REGION,STATUT REGION,STATUT VILLE_CL_Région,STATUT VILLE_CL_DR,Taille,Activite,Strate,Agent,Commune,Entrée,Effectif_Employé,CA22_corrige
0,1715838X,SOCIETE DE DEVELOPPEMENT DU CAOUTCHOUC IVOIRIEN,"ABIDJAN-COCODY-RUE LYCEE TECHNIQUE, S/N RESIDE...",SOCIETE DE DEVELOPPEMENT DU CAOUTCHOUC IVOIRIEN,2722406135,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Agriculture,Strate 1,AKE Akalé Florence,COCODY,Echantillon_1,763.0,8.929433e+10
1,0815218H,OFFICE NATIONAL DE L'EAU POTABLE,COCODY II PLATEAUX VALLONS RUE J 93,OFFICE NATIONAL DE L'EAU POTABLE,2722514300,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5,AKE Akalé Florence,COCODY,Echantillon_3,270.0,8.327025e+09
2,1942796Z,TECHNIC BETON,KM 20 AUTOROUTE DU NORD ANYAMA ALLOKOI 08 BP 4...,TECHNIC BETON,0707111111,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Industrie Manufacturière,Strate 3,AKE Akalé Florence,ABOBO,Echantillon_3,327.0,4.574771e+09
3,4104288G,SOCIETE DE DISTRIBUTION ET DESTRAVAUX,ABIDJAN -COCODY -RIVIERA - AKOUEDO PALMERAIE,SOCIETE DE DISTRIBUTION ET DESTRAVAUX,0778542914,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5,AKE Akalé Florence,COCODY,Echantillon_2,539.0,1.818725e+10
4,9801444C,SOCIETE CONCESSIONNAIRE DU PONT RIVIERA MARCORY,"Bâtiment opérationnel, sis à la barrière de pé...",SOCIETE CONCESSIONNAIRE DU PONT RIVIERA MARCORY,2722427300,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Autres Industries,Strate 5,AKE Akalé Florence,COCODY,Echantillon_2,208.0,1.818120e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1791,1715354H,TERMINAL INDUSTRIEL POLYVALENT DE SAN PEDRO,"San Pedro, Boulevard Zone d¿extension du Port,...",TERMINAL INDUSTRIEL POLYVALENT DE SAN PEDRO,2722505950 / 2722505905 / 2722505905 / 2722505906,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Service,Strate 9,TCHESSIAN Roland-Franck,ABIDJAN,Echantillon_1,100.0,3.406773e+10
1792,0332868Q,INSTITUT TRAORE ADAMA,KOUMASSI GRAND MARCHE,INSTITUT TRAORE ADAMA,0504102222,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,PME,Service,Strate 10,TCHESSIAN Roland-Franck,KOUMASSI,Echantillon_5,11.0,4.925000e+07
1793,1321738F,ATLANTIC BUSINESS INTERNATIONA,TOUR AMCI 15 AVENUE JOSEPH ANOMA COTE D'IVOIRE,ATLANTIC BUSINESS INTERNATIONA,0103141400,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,GE,Service,Strate 9,TCHESSIAN Roland-Franck,ABIDJAN,Echantillon_1,212.0,2.063906e+10
1794,0027449E,ZAHHAR EPSE FAZAA Marie Therese,"IMMEUBLE LE MASSAI, Bd VGE, MARCORY/ABIDJAN",ZAHHAR EPSE FAZAA Marie Therese,2721246056,ABIDJAN,ABIDJAN,ABIDJAN,ABIDJAN,Région de résidence de la DR,Ville est CL de région,Ville est CL de DR,PME,Service,Strate 10,TCHESSIAN Roland-Franck,MARCORY,Echantillon_5,2.0,4.722400e+07


In [104]:
def good_div(x):
    for i in range(4):
        if round(x,i)!=0 and i!=0:
            return str(round(x,i))+'%'
        elif round(x,i)!=0 and i==0:
            return str(int(round(x)))+'%'
    else:
        return '0%'

In [108]:
t=pd.DataFrame(sorted(da.DR.unique()),columns=['DR'])
t['Nbre']=t.DR.apply(lambda x: len(da.loc[(da.DR==x)&(da['STATUT VILLE_CL_DR']!='Ville est CL de DR')]))
t['Poids1']=t.Nbre.apply(lambda x: good_div(100*x/t.Nbre.sum()))
t['Effectif']=t.DR.apply(lambda x: da.loc[(da.DR==x)&(da['STATUT VILLE_CL_DR']!='Ville est CL de DR')].Effectif_Employé.sum())
t['Poids2']=t.Effectif.apply(lambda x: good_div(100*x/t.Effectif.sum()))
t['CA']=t.DR.apply(lambda x: da.loc[(da.DR==x)&(da['STATUT VILLE_CL_DR']!='Ville est CL de DR')].CA22_corrige.sum())
t['Poids3']=t.CA.apply(lambda x: good_div(100*x/t.CA.sum()))
t['CA']=t.CA.apply(lambda x: mont(x))
t

,DR,Nbre,Poids1,Effectif,Poids2,CA,Poids3
0,ABENGOUROU,107,37%,5951.0,18%,231 Milliards,20%
1,ABIDJAN,30,10%,4643.0,14%,159 Milliards,14%
2,AGBOVILLE,20,7%,3976.0,12%,29 Milliards,3%
3,BONDOUKOU,5,2%,135.0,0.4%,276 Millions,0.02%
4,BOUAKE,7,2%,1948.0,6%,208 Milliards,18%
5,DALOA,10,3%,271.0,1%,32 Milliards,3%
6,DAOUKRO,5,2%,309.0,1%,7 Milliards,1%
7,GAGNOA,10,3%,361.0,1%,26 Milliards,2%
8,KORHOGO,26,9%,7096.0,22%,180 Milliards,16%
9,MAN,9,3%,370.0,1%,30 Milliards,3%


In [124]:
t=pd.DataFrame(sorted(da.Strate.unique()),columns=['Strate'])
t['Nbre']=t.Strate.apply(lambda x: len(da.loc[(da.Strate==x)&(da['STATUT VILLE_CL_DR']!='Ville est CL de DR')]))
t['Poids1']=t.Nbre.apply(lambda x: good_div(100*x/t.Nbre.sum()))
t['Effectif']=t.Strate.apply(lambda x: da.loc[(da.Strate==x)&(da['STATUT VILLE_CL_DR']!='Ville est CL de DR')].Effectif_Employé.sum())
t['Poids2']=t.Effectif.apply(lambda x: good_div(100*x/t.Effectif.sum()))
t['CA']=t.Strate.apply(lambda x: da.loc[(da.Strate==x)&(da['STATUT VILLE_CL_DR']!='Ville est CL de DR')].CA22_corrige.sum())
t['Poids3']=t.CA.apply(lambda x: good_div(100*x/t.CA.sum()))
t['CA']=t.CA.apply(lambda x: mont(x))
t

,Strate,Nbre,Poids1,Effectif,Poids2,CA,Poids3
0,Strate 1,11,4%,12289.0,37%,193 Milliards,17%
1,Strate 10,94,32%,2428.0,7%,15 Milliards,1%
2,Strate 2,17,6%,285.0,1%,3 Milliards,0.3%
3,Strate 3,24,8%,9563.0,29%,438 Milliards,39%
4,Strate 4,13,4%,287.0,1%,2 Milliards,0.2%
5,Strate 5,6,2%,1013.0,3%,44 Milliards,4%
6,Strate 6,7,2%,34.0,0.1%,2 Milliards,0.2%
7,Strate 7,39,13%,2623.0,8%,301 Milliards,27%
8,Strate 8,58,20%,365.0,1%,33 Milliards,3%
9,Strate 9,24,8%,3926.0,12%,94 Milliards,8%


In [139]:
da.Entrée.value_counts()

Entrée
Echantillon_6    319
Echantillon_5    318
Echantillon_1    302
Echantillon_4    296
Echantillon_3    284
Echantillon_2    277
Name: count, dtype: int64

In [162]:
(302)/15

20.133333333333333